"""
Ingesta: carga los archivos fuente en el esquema `raw` de warehouse.duckdb.

Estrategia: full refresh (CREATE OR REPLACE). La fuente es un snapshot completo,
sin updated_at ni CDC.
- CSV: todo como VARCHAR. El casteo y la validación se hacen en dbt (staging).
- products.json: un registro por producto, tal cual viene.
- fx_rates.json: una fila por respuesta de la API; `rates` se guarda como JSON
  para no depender de qué monedas trae cada respuesta.
"""

In [ ]:
from pathlib import Path
import duckdb

ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "analytics_engineer_assets"
DB = ROOT / "warehouse.duckdb"

CSV_SOURCES = {
    "customers": "customers.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
}


def load_csv(con, table, fname):
    path = (DATA / fname).as_posix()
    con.execute(f"""
        CREATE OR REPLACE TABLE raw.{table} AS
        SELECT *,
               current_timestamp AS _loaded_at,
               '{fname}'         AS _source_file
        FROM read_csv('{path}', header = true, all_varchar = true)
    """)


def load_products(con):
    path = (DATA / "products.json").as_posix()
    con.execute(f"""
        CREATE OR REPLACE TABLE raw.products AS
        SELECT *,
               current_timestamp AS _loaded_at,
               'products.json'   AS _source_file
        FROM read_json_auto('{path}')
    """)


def load_fx_rates(con):
    path = (DATA / "fx_rates.json").as_posix()
    con.execute(f"""
        CREATE OR REPLACE TABLE raw.fx_rates AS
        WITH responses AS (
            SELECT unnest(from_json(json(content) -> 'responses', '["JSON"]')) AS resp
            FROM read_text('{path}')
        )
        SELECT resp ->> 'base'    AS base_currency,
               resp ->> 'date'    AS rate_date,
               resp -> 'rates'    AS rates,
               current_timestamp  AS _loaded_at,
               'fx_rates.json'    AS _source_file
        FROM responses
    """)


def main():
    con = duckdb.connect(str(DB))
    con.execute("CREATE SCHEMA IF NOT EXISTS raw")

    for table, fname in CSV_SOURCES.items():
        load_csv(con, table, fname)
    load_products(con)
    load_fx_rates(con)

    for (table,) in con.execute(
        "SELECT table_name FROM information_schema.tables "
        "WHERE table_schema = 'raw' ORDER BY 1"
    ).fetchall():
        n = con.execute(f"SELECT count(*) FROM raw.{table}").fetchone()[0]
        print(f"raw.{table}: {n} filas")

    con.close()


if __name__ == "__main__":
    main()